In [7]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from torchvision import io
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchvision import transforms
import matplotlib.pyplot as plt

In [8]:

def min_max_normalize(tensor, min_val=0.0, max_val=1.0):
    tensor_min = tensor.min()
    tensor_max = tensor.max()
    normalized_tensor = (tensor - tensor_min) / (tensor_max - tensor_min) * (max_val - min_val) + min_val
    return normalized_tensor

def resize_torch_tensor(tensor, w=256, h=256):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((w, h)),
    ])
    tensor = transform(tensor)
    tensor = tensor.float()

    tensor = min_max_normalize(tensor)

    return tensor

In [17]:
class RealAndSyntheticImageDataset(torch.utils.data.Dataset):
    def __init__(self, num_samples, synthetic_images_folder_path, real_images_folder_path):
        synthetic_images = os.listdir(synthetic_images_folder_path)

        real_images = os.listdir(real_images_folder_path)

        if len(synthetic_images) < num_samples:
            raise ValueError(f"Not enough synthetic images. Found {len(synthetic_images)}, but requested {num_samples}.")
        
        if len(real_images) < num_samples:
            raise ValueError(f"Not enough real images. Found {len(real_images)}, but requested {num_samples}.")
        
        synthetic_images_paths = [os.path.join(synthetic_images_folder_path, img) for img in synthetic_images[:(num_samples // 2)]]
        real_images_paths = [os.path.join(real_images_folder_path, img) for img in real_images[:(num_samples // 2)]]

        self.paths = []
        self.labels = []
        # Interleave the paths
        for i in range(min(len(synthetic_images_paths), len(real_images_paths))):
            self.paths.append(synthetic_images_paths[i])
            self.paths.append(real_images_paths[i])
            self.labels.append(1)
            self.labels.append(0)

        self.length = len(self.paths)

    def __len__(self):
        return self.length
    def __getitem__(self, idx):
        # Return a random 1×266×266 tensor and a random binary label
        image_path = self.paths[idx]
        image_data = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        # Resize and normalize the loaded data
        image_data = resize_torch_tensor(image_data)

        label = self.labels[idx]
        
        return image_data, label

In [39]:
dataset = RealAndSyntheticImageDataset(
    10, 'singan_output_from_similarity', '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/Input/data-RGBA'
)
train_loader = DataLoader(dataset, batch_size=1, shuffle=True)

In [42]:
for image, label in train_loader:
    print(image.shape, label)
    # Here you can add code to visualize the image if needed
    # break  # Remove this line to iterate through the entire dataset

torch.Size([1, 1, 256, 256]) tensor([0])
torch.Size([1, 1, 256, 256]) tensor([0])
torch.Size([1, 1, 256, 256]) tensor([0])
torch.Size([1, 1, 256, 256]) tensor([0])
torch.Size([1, 1, 256, 256]) tensor([1])
torch.Size([1, 1, 256, 256]) tensor([1])
torch.Size([1, 1, 256, 256]) tensor([1])
torch.Size([1, 1, 256, 256]) tensor([0])
torch.Size([1, 1, 256, 256]) tensor([1])
torch.Size([1, 1, 256, 256]) tensor([1])


In [27]:
label

tensor([0, 0, 1, 0, 0, 1, 1, 1, 1, 0])